# BSAN 391 - Transshipment (Min Cost Network Flow) - Redbrand - Gurobi

In [1]:
from gurobipy import *

In [2]:
location, demand, supply = multidict({1: [0, 200], 2: [0, 300], 3: [0, 100], 4: [0, 0], 5: [0, 0], 6: [400, 0], 7: [180, 0]})
arcs, cost = multidict({
    (1,2): 5,
    (1,3): 3,
    (1,4): 5,
    (1,5): 5,
    (1,6): 20,
    (1,7): 20,
    (2,1): 9,
    (2,3): 9,
    (2,4): 1,
    (2,5): 1,
    (2,6): 8,
    (2,7): 15,
    (3,1): 0.4,
    (3,2): 8,
    (3,4): 1,
    (3,5): 0.5,
    (3,6): 10,
    (3,7): 12,
    (4,5): 1.2,
    (4,6): 2,
    (4,7): 12,
    (5,4): 0.8,
    (5,6): 2,
    (5,7): 12,
    (6,7): 1,
    (7,6): 7
})
arcCapacity = 200

In [3]:
mincostflow = Model("Transshipment")

Academic license - for non-commercial use only - expires 2022-08-04
Using license file C:\Users\novoadlj\gurobi.lic


In [4]:
flow = mincostflow.addVars(arcs, name="flow")
totalcost = mincostflow.setObjective(quicksum(flow[i,j]*cost[i,j] for (i,j) in arcs))

In [5]:
balance = mincostflow.addConstrs(quicksum(flow[j,i] for j in location if (j, i) in arcs)
                                 - quicksum(flow[i,j] for j in location if (i, j) in arcs) >= (demand[i] - supply[i]) for i in location)
arccap = mincostflow.addConstrs(flow[i,j] <= arcCapacity for (i,j) in arcs)

In [6]:
mincostflow.optimize()

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 33 rows, 26 columns and 78 nonzeros
Model fingerprint: 0xffddead4
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 4e+02]
Presolve removed 26 rows and 0 columns
Presolve time: 0.01s
Presolved: 7 rows, 26 columns, 52 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   7.250000e+01   0.000000e+00      0s
       7    3.2600000e+03   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds
Optimal objective  3.260000000e+03


In [7]:
# Display optimal production plan
for v in mincostflow.getVars():
    if v.x>0:
        print(v.varName, v.x)
    
print("Optimal total shipping cost:", "$"+str(mincostflow.objVal))

flow[1,3] 180.0
flow[2,4] 120.0
flow[2,6] 180.0
flow[3,4] 80.0
flow[3,5] 200.0
flow[4,6] 200.0
flow[5,6] 200.0
flow[6,7] 180.0
Optimal total shipping cost: $3260.0
